# AI-Powered Retail Analytics Assistant
### AI Engineer Assessment – OneTapp Consulting

## 1. Install Dependencies

In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import Libraries

In [2]:
import os
from dotenv import load_dotenv
import sqlite3
import warnings

import pandas as pd

from huggingface_hub import InferenceClient

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

from langchain_text_splitters import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore")

load_dotenv()

d:\OneTap Assessment\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## 3. Project Configuration

In [45]:
DATASET_PATH = "data/sales_data.csv"
KNOWLEDGE_PATH = "data/knowledge.md"

DATABASE_PATH = "database/sales.db"

TABLE_NAME = "sales_data"

HF_API_KEY = os.getenv("HF_API_KEY")
HF_API_KEY2 = os.getenv("HF_API_KEY2")

HF_MODEL = "google/gemma-4-31B-it"

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

CHROMA_COLLECTION = "retail_knowledge"

## 4. Verify Project Files

In [4]:
assert os.path.exists(DATASET_PATH), "sales_data.csv not found."
assert os.path.exists(KNOWLEDGE_PATH), "knowledge.md not found."

print("✓ Files found")

✓ Files found


# Data Layer

## 5. Load Sales Dataset

In [5]:
# Load dataset

sales_df = pd.read_csv(DATASET_PATH)

print(f"Rows    : {sales_df.shape[0]}")
print(f"Columns : {sales_df.shape[1]}")

sales_df.head()

Rows    : 11978
Columns : 19


,transaction_id,week_start,region,state,city,store_id,product_id,product_name,category,brand,promotion_name,promotion_type,promotion_active,units_sold,revenue,discount_amount,inventory_before,inventory_after,customer_rating
0,100001,2026-01-05,North,Delhi,New Delhi,STR101,P114,Harpic 500ml,Household,Reckitt,NaN,NaN,False,43,4337.99,0.0,867,824,4.2
1,100002,2026-01-05,North,Delhi,New Delhi,STR101,P108,Amul Milk 1L,Dairy,Amul,NaN,NaN,False,107,7102.64,0.0,880,773,4.7
2,100003,2026-01-05,North,Delhi,New Delhi,STR101,P104,Lay's Classic 52g,Snacks,PepsiCo,NaN,NaN,False,103,2087.00,0.0,638,535,4.3
3,100004,2026-01-05,North,Delhi,New Delhi,STR101,P111,Colgate Toothpaste 150g,Personal Care,Colgate,NaN,NaN,False,72,6045.25,0.0,815,743,4.3
4,100005,2026-01-05,North,Delhi,New Delhi,STR101,P109,Nestle Milkmaid 400g,Dairy,Nestle,NaN,NaN,False,32,3096.85,0.0,705,673,4.2


## 6. Create SQLite Database

In [6]:
# Create database directory if it doesn't exist

os.makedirs("database", exist_ok=True)

# Create SQLite connection

conn = sqlite3.connect(DATABASE_PATH)

# Write dataframe to SQLite

sales_df.to_sql(
    TABLE_NAME,
    conn,
    if_exists="replace",
    index=False
)

print("✓ SQLite database created successfully")

✓ SQLite database created successfully


## 7. Verify Database Creation

In [7]:
query = f"""
SELECT *
FROM {TABLE_NAME}
LIMIT 5;
"""

pd.read_sql(query, conn)

,transaction_id,week_start,region,state,city,store_id,product_id,product_name,category,brand,promotion_name,promotion_type,promotion_active,units_sold,revenue,discount_amount,inventory_before,inventory_after,customer_rating
0,100001,2026-01-05,North,Delhi,New Delhi,STR101,P114,Harpic 500ml,Household,Reckitt,None,None,0,43,4337.99,0.0,867,824,4.2
1,100002,2026-01-05,North,Delhi,New Delhi,STR101,P108,Amul Milk 1L,Dairy,Amul,None,None,0,107,7102.64,0.0,880,773,4.7
2,100003,2026-01-05,North,Delhi,New Delhi,STR101,P104,Lay's Classic 52g,Snacks,PepsiCo,None,None,0,103,2087.00,0.0,638,535,4.3
3,100004,2026-01-05,North,Delhi,New Delhi,STR101,P111,Colgate Toothpaste 150g,Personal Care,Colgate,None,None,0,72,6045.25,0.0,815,743,4.3
4,100005,2026-01-05,North,Delhi,New Delhi,STR101,P109,Nestle Milkmaid 400g,Dairy,Nestle,None,None,0,32,3096.85,0.0,705,673,4.2


## 8. Validate Dataset Integrity

In [8]:
query = f"""
SELECT COUNT(*) AS total_rows
FROM {TABLE_NAME};
"""

pd.read_sql(query, conn)

,total_rows
0,11978


## 9. Test SQL Execution

In [9]:
query = f"""
SELECT
    region,
    SUM(revenue) AS total_revenue
FROM {TABLE_NAME}
GROUP BY region
ORDER BY total_revenue DESC;
"""

pd.read_sql(query, conn)

,region,total_revenue
0,South,10724693.11
1,North,10206520.72
2,Central,7893047.29
3,West,7813584.22
4,East,7621674.41


# Dynamic Schema Discovery

## 10. Implement Schema Inspector

In [10]:
from typing import Dict, List


from typing import List
import pandas as pd


class SchemaInspector:
    def __init__(self, connection):
        self.conn = connection

    def get_tables(self) -> List[str]:
        query = """
        SELECT name
        FROM sqlite_master
        WHERE type='table';
        """
        tables = pd.read_sql(query, self.conn)
        return tables["name"].tolist()

    def get_columns(self, table_name: str) -> pd.DataFrame:
        return pd.read_sql(f"PRAGMA table_info({table_name});", self.conn)

    def get_sample_values(self, table_name: str, column: str, limit: int = 5):
        query = f"""
        SELECT DISTINCT "{column}"
        FROM {table_name}
        WHERE "{column}" IS NOT NULL
        LIMIT {limit};
        """
        values = pd.read_sql(query, self.conn)
        return values[column].tolist()

    def generate_schema_summary(self, table_name: str):

        columns = self.get_columns(table_name)

        total_rows = pd.read_sql(
            f"SELECT COUNT(*) AS cnt FROM {table_name}",
            self.conn
        )["cnt"][0]

        summary = []
        summary.append("=" * 60)
        summary.append("DATABASE SCHEMA")
        summary.append("=" * 60)
        summary.append(f"Table Name      : {table_name}")
        summary.append(f"Total Rows      : {total_rows}")
        summary.append(f"Total Columns   : {len(columns)}")
        summary.append("")

        summary.append("Columns")
        summary.append("-" * 60)

        for _, row in columns.iterrows():

            column = row["name"]
            dtype = row["type"]

            null_count = pd.read_sql(
                f'''
                SELECT COUNT(*)
                FROM {table_name}
                WHERE "{column}" IS NULL
                ''',
                self.conn
            ).iloc[0, 0]

            unique_count = pd.read_sql(
                f'''
                SELECT COUNT(DISTINCT "{column}")
                FROM {table_name}
                ''',
                self.conn
            ).iloc[0, 0]

            samples = self.get_sample_values(table_name, column)

            summary.append(f"Column : {column}")
            summary.append(f"Type   : {dtype}")
            summary.append(f"Unique : {unique_count}")
            summary.append(f"Nulls  : {null_count}")
            summary.append(f"Samples: {samples}")

            if dtype.upper() in ["INTEGER", "REAL", "FLOAT", "NUMERIC"]:

                stats = pd.read_sql(
                    f'''
                    SELECT
                        MIN("{column}") AS min_val,
                        MAX("{column}") AS max_val
                    FROM {table_name}
                    ''',
                    self.conn
                )

                summary.append(f"Min    : {stats['min_val'][0]}")
                summary.append(f"Max    : {stats['max_val'][0]}")

            summary.append("")

        return "\n".join(summary)

## 11. Initialize Schema Inspector

In [11]:
schema = SchemaInspector(conn)

## 12. Discover Database Tables

In [12]:
tables = schema.get_tables()

tables

['sales_data']

## 13. Inspect Table Schema

In [13]:
columns = schema.get_columns(TABLE_NAME)

columns

,cid,name,type,notnull,dflt_value,pk
0,0,transaction_id,INTEGER,0,None,0
1,1,week_start,TEXT,0,None,0
2,2,region,TEXT,0,None,0
3,3,state,TEXT,0,None,0
4,4,city,TEXT,0,None,0
5,5,store_id,TEXT,0,None,0
6,6,product_id,TEXT,0,None,0
7,7,product_name,TEXT,0,None,0
8,8,category,TEXT,0,None,0
9,9,brand,TEXT,0,None,0


## 14. Explore Sample Values

In [14]:
for column in ["region", "category", "brand", "promotion_type"]:
    print(f"\n{column}")
    print(schema.get_sample_values(TABLE_NAME, column))


region
['North', 'South', 'East', 'West', 'Central']

category
['Household', 'Dairy', 'Snacks', 'Personal Care', 'Soft Drinks']

brand
['Reckitt', 'Amul', 'PepsiCo', 'Colgate', 'Nestle']

promotion_type
['Flat Discount', 'BOGO', 'Percentage Discount']


## 15. Generate Schema Context

In [15]:
schema_context = schema.generate_schema_summary(TABLE_NAME)

print(schema_context)

DATABASE SCHEMA
Table Name      : sales_data
Total Rows      : 11978
Total Columns   : 19

Columns
------------------------------------------------------------
Column : transaction_id
Type   : INTEGER
Unique : 11978
Nulls  : 0
Samples: [100001, 100002, 100003, 100004, 100005]
Min    : 100001
Max    : 111978

Column : week_start
Type   : TEXT
Unique : 26
Nulls  : 0
Samples: ['2026-01-05', '2026-01-12', '2026-01-19', '2026-01-26', '2026-02-02']

Column : region
Type   : TEXT
Unique : 5
Nulls  : 0
Samples: ['North', 'South', 'East', 'West', 'Central']

Column : state
Type   : TEXT
Unique : 13
Nulls  : 0
Samples: ['Delhi', 'Punjab', 'Uttar Pradesh', 'Karnataka', 'Tamil Nadu']

Column : city
Type   : TEXT
Unique : 17
Nulls  : 0
Samples: ['New Delhi', 'Ludhiana', 'Lucknow', 'Agra', 'Bangalore']

Column : store_id
Type   : TEXT
Unique : 34
Nulls  : 0
Samples: ['STR101', 'STR102', 'STR103', 'STR104', 'STR105']

Column : product_id
Type   : TEXT
Unique : 20
Nulls  : 0
Samples: ['P114', 'P108', 

## 16. Save Schema Context

In [16]:
os.makedirs("database", exist_ok=True)

with open("database/schema_context.txt", "w", encoding="utf-8") as f:
    f.write(schema_context)

print("✓ Schema context saved successfully.")

✓ Schema context saved successfully.


# Knowledge Base

## 17. Load Knowledge Base

In [17]:
with open(KNOWLEDGE_PATH, "r", encoding="utf-8") as f:
    knowledge_text = f.read()

print(f"Knowledge Base Loaded ({len(knowledge_text)} characters)")

Knowledge Base Loaded (9106 characters)


## 18. Split Knowledge Base into Chunks

In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "]
)

knowledge_chunks = text_splitter.split_text(knowledge_text)

print(f"Total Chunks: {len(knowledge_chunks)}")

Total Chunks: 31


## 19. Preview Knowledge Chunks

In [19]:
for i, chunk in enumerate(knowledge_chunks[:3]):
    print("=" * 80)
    print(f"Chunk {i+1}")
    print("=" * 80)
    print(chunk)
    print()

Chunk 1
# Knowledge Base: Retail Sales Analytics (sales_data)

This document defines every metric, business rule, and dimension used in the
`sales_data` table. It is written for retrieval — each section is a
self-contained fact block so a RAG system can pull the right definition
without needing surrounding context.

---

## Table Grain

One row = one product sold in one region during one week.
Primary key: `transaction_id`.
Time grain: weekly, anchored to `week_start` (always a Monday).

---

Chunk 2
## Metrics

Chunk 3
### Promotion Lift

Promotion Lift =
(Current Promotion Revenue - Baseline Revenue) / Baseline Revenue



## 20. Initialize ChromaDB

In [20]:
embedding_function = SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL
)

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name=CHROMA_COLLECTION,
    embedding_function=embedding_function
)

print("✓ ChromaDB initialized")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1884.68it/s]


✓ ChromaDB initialized


## 21. Index Knowledge Chunks

In [21]:
collection.add(
    documents=knowledge_chunks,
    ids=[f"chunk_{i}" for i in range(len(knowledge_chunks))]
)

print(f"✓ Indexed {len(knowledge_chunks)} chunks")

✓ Indexed 31 chunks


## 22. Test Semantic Retrieval

In [22]:
query = "What is Promotion Lift?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

for i, document in enumerate(results["documents"][0], start=1):
    print("=" * 80)
    print(f"Retrieved Chunk {i}")
    print("=" * 80)
    print(document)
    print()

Retrieved Chunk 1
### Promotion Lift

Promotion Lift =
(Current Promotion Revenue - Baseline Revenue) / Baseline Revenue

Retrieved Chunk 2
- Positive lift = promotion increased revenue vs. its own pre-promotion
  baseline. Negative lift = revenue fell relative to baseline.

Retrieved Chunk 3
## Promotion Types



# Large Language Model

## 23. Initialize Hugging Face Inference Client

In [47]:
from huggingface_hub import InferenceClient

client = InferenceClient(
    api_key=HF_API_KEY
)

client2 = InferenceClient(
    api_key=HF_API_KEY2
)

print("✓ Hugging Face clients initialized")

✓ Hugging Face clients initialized


## 24. Configure the LLM

In [24]:
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"

TEMPERATURE = 0.1
MAX_TOKENS = 1024
TOP_P = 0.9

print(f"Model: {LLM_MODEL}")

Model: Qwen/Qwen2.5-7B-Instruct


## 25. Test the LLM Connection

In [25]:
response_test = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Reply with only the word 'Connected'."
        }
    ],
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    top_p=TOP_P
)

print(response_test.choices[0].message.content)

Connected


# Query Processing Pipeline

## 26. Implement Query Planner

In [26]:
import json


class QueryPlanner:
    def __init__(self, client, model):
        self.client = client
        self.model = model

    def plan(self, user_query: str):

        prompt = f"""
You are an AI planning agent for a Retail Analytics Assistant.

Your job is NOT to answer the user's question.

Instead, decide which tools are required and produce a plan.

Available Tools:

1. SQL Database
- Used for numerical analysis.
- Sales trends.
- Revenue.
- Units sold.
- Profit.
- Customer ratings.
- Product performance.
- Regional performance.
- Any information present in the sales database.

2. Knowledge Base
- Used for business concepts.
- Retail KPIs.
- Promotion definitions.
- Formula explanations.
- Business rules.
- Domain knowledge.

Instructions:

1. Decide whether SQL is required.
2. Decide whether the Knowledge Base is required.
3. If SQL is needed, rewrite the user's question into a clear SQL objective.
4. Do NOT generate SQL statements.
5. Only describe the analytical objective that the SQL generator should perform.
6. If the Knowledge Base is needed, rewrite the user's question into a retrieval objective.
7. Provide a short description of the final answer that should be generated.
8. Return ONLY valid JSON.
9. Never wrap the JSON inside markdown.

Return EXACTLY this format:

{{
    "needs_sql": true,
    "needs_rag": false,
    "sql_task": "",
    "rag_task": "",
    "answer_goal": ""
}}

User Question:
{user_query}
"""

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            max_tokens=300
        )

        return json.loads(response.choices[0].message.content)

## 27. Initialize Query Planner

In [48]:
planner = QueryPlanner(
    client2,
    LLM_MODEL
)

## 28. Test Query Planner

In [28]:
queries = [
    "Which region generated the highest revenue?",
    "What is Promotion Lift?",
    "Explain whether promotions improved sales in the South region."
]

for query in queries:
    print("=" * 80)
    print(query)
    print("=" * 80)

    plan = planner.plan(query)

    print(json.dumps(plan, indent=4))
    print()

Which region generated the highest revenue?
{
    "needs_sql": true,
    "needs_rag": false,
    "sql_task": "Determine the region with the highest total revenue.",
    "rag_task": "",
    "answer_goal": "The region that generated the highest revenue."
}

What is Promotion Lift?
{
    "needs_sql": false,
    "needs_rag": true,
    "sql_task": "",
    "rag_task": "Retrieve the definition and explanation of Promotion Lift from the Knowledge Base.",
    "answer_goal": "A definition and explanation of what Promotion Lift is in the context of retail analytics."
}

Explain whether promotions improved sales in the South region.
{
    "needs_sql": true,
    "needs_rag": true,
    "sql_task": "Calculate the total sales and number of promotions in the South region before and after a specific period to compare sales trends.",
    "rag_task": "Define the criteria for what constitutes a promotion and explain how to measure its impact on sales in the South region.",
    "answer_goal": "An analysis i

## 29. Implement SQL Generator

In [29]:
class SQLGenerator:
    def __init__(self, client, model, schema_context):
        self.client = client
        self.model = model
        self.schema_context = schema_context

    def generate_sql(self, sql_task):

        prompt = f"""
You are an expert SQLite SQL generator.

Database Schema:

{self.schema_context}

Your task is to convert the user's question into a valid SQLite query.

Rules:
1. Return ONLY valid JSON.
2. Do NOT wrap the JSON in markdown.
3. Do NOT include explanations outside the JSON.
4. Generate ONLY SQLite-compatible SQL.
5. Use ONLY tables and columns present in the schema.
6. Never use SELECT *.
7. Always use explicit column names.
8. Always alias aggregated columns.
9. Include LIMIT only when appropriate.
10. Generate ONLY read-only SELECT queries.
11. Never generate INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, TRUNCATE, or PRAGMA statements.

Return JSON in exactly this format:

{{
    "sql": "<generated_sql>",
    "summary": "<one sentence describing what the query does>"
}}


SQL Objective:
{sql_task}
"""

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            max_tokens=400
        )

        return json.loads(response.choices[0].message.content)

## 30. Initialize SQL Generator

In [49]:
sql_generator = SQLGenerator(
    client2,
    LLM_MODEL,
    schema_context
)

## 31. Test SQL Generation

In [31]:
question = "Which region generated the highest revenue?"

response_sql = sql_generator.generate_sql(question)

print(response_sql)

{'sql': 'SELECT region, SUM(revenue) AS total_revenue FROM sales_data GROUP BY region ORDER BY total_revenue DESC LIMIT 1;', 'summary': 'This query calculates the total revenue for each region and returns the region with the highest revenue.'}


## 32. Implement SQL Executor

In [32]:
class SQLExecutor:
    def __init__(self, connection):
        self.conn = connection

    def execute(self, sql_query):
        try:
            result = pd.read_sql(sql_query, self.conn)
            return result

        except Exception as e:
            return f"SQL Execution Error: {str(e)}"

## 33. Initialize SQL Executor

In [33]:
sql_executor = SQLExecutor(conn)

## 34. Execute Generated SQL

In [34]:
generated_sql = response_sql["sql"]

query_result = sql_executor.execute(generated_sql)

display(query_result)

,region,total_revenue
0,South,10724693.11


# RAG

## 35. Implement Knowledge Retriever

In [35]:
class KnowledgeRetriever:
    def __init__(self, collection):
        self.collection = collection

    def retrieve(self, query, top_k=3):
        results = self.collection.query(
            query_texts=[query],
            n_results=top_k
        )

        return results["documents"][0]

## 36. Initialize Knowledge Retriever

In [36]:
knowledge_retriever = KnowledgeRetriever(collection)

## 37. Test Knowledge Retrieval

In [37]:
query = "What is Promotion Lift?"

documents = knowledge_retriever.retrieve(query)

for i, doc in enumerate(documents, start=1):
    print("=" * 80)
    print(f"Retrieved Document {i}")
    print("=" * 80)
    print(doc)
    print()

Retrieved Document 1
### Promotion Lift

Promotion Lift =
(Current Promotion Revenue - Baseline Revenue) / Baseline Revenue

Retrieved Document 2
- Positive lift = promotion increased revenue vs. its own pre-promotion
  baseline. Negative lift = revenue fell relative to baseline.

Retrieved Document 3
## Promotion Types



# Response Generator

## 38. Implement Response Generator

In [38]:
class ResponseGenerator:
    def __init__(self, client, model):
        self.client = client
        self.model = model

    def generate(
    self,
    user_query,
    answer_goal,
    sql_result=None,
    retrieved_context=None
    ):

        prompt = f"""
You are a senior Retail Business Analyst.

Answer the user's question professionally.

User Question:
{user_query}

Answer Goal:
{answer_goal}

SQL Result:
{sql_result}

Knowledge Context:
{retrieved_context}

Rules:
1. Base your answer ONLY on the provided information.
2. Never invent facts.
3. If information is missing, explicitly say so.
4. Explain insights in simple business language.
5. Keep the answer concise.
"""

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            max_tokens=500
        )

        return response.choices[0].message.content

## 39. Initialize Response Generator

In [50]:
response_generator = ResponseGenerator(
    client,
    LLM_MODEL
)

## 40. Test Response Generation (SQL)

In [40]:
answer = response_generator.generate(
    user_query=question,
    answer_goal="Identify the region with the highest revenue.",
    sql_result=query_result.to_string(index=False),
    retrieved_context=""
)

print(answer)

Based on the provided information, the South region generated the highest revenue with a total of $10,724,693.11.


## 41. Test Response Generation (RAG)

In [41]:
documents = knowledge_retriever.retrieve(
    "What is Promotion Lift?"
)

context = "\n\n".join(documents)

answer = response_generator.generate(
    user_query="What is Promotion Lift?",
    answer_goal="Explain the concept of Promotion Lift.",
    sql_result="",
    retrieved_context=context
)

print(answer)

Promotion Lift is a metric used to measure the effectiveness of a promotional activity. It helps retailers understand how much additional revenue a promotion has generated compared to what would have been expected without the promotion.

The formula for calculating Promotion Lift is:

\[ \text{Promotion Lift} = \frac{\text{Current Promotion Revenue} - \text{Baseline Revenue}}{\text{Baseline Revenue}} \]

- **Positive lift** indicates that the promotion has increased revenue compared to the baseline (expected revenue without the promotion).
- **Negative lift** indicates that the promotion has resulted in lower revenue than expected.

In simple terms, Promotion Lift helps you determine whether a promotion has been successful in driving additional sales.


# AI Retail Analytics Assistant

## 42. Implement AI Retail Analytics Assistant

In [42]:
class AIRetailAnalyticsAssistant:

    def __init__(
        self,
        planner,
        sql_generator,
        sql_executor,
        knowledge_retriever,
        response_generator
    ):

        self.planner = planner
        self.sql_generator = sql_generator
        self.sql_executor = sql_executor
        self.knowledge_retriever = knowledge_retriever
        self.response_generator = response_generator

    def ask(self, user_query):

        plan = self.planner.plan(user_query)

        print(json.dumps(plan, indent=4))

        sql_result = ""
        retrieved_context = ""

        # ---------- SQL ----------
        if plan["needs_sql"]:

            sql_output = self.sql_generator.generate_sql(
                plan["sql_task"]
            )

            generated_sql = sql_output["sql"]

            dataframe = self.sql_executor.execute(generated_sql)

            sql_result = dataframe.to_string(index=False)

        # ---------- RAG ----------
        if plan["needs_rag"]:

            documents = self.knowledge_retriever.retrieve(
                plan["rag_task"]
            )

            retrieved_context = "\n\n".join(documents)

        # ---------- Final Response ----------
        return self.response_generator.generate(
            user_query=user_query,
            answer_goal=plan["answer_goal"],
            sql_result=sql_result,
            retrieved_context=retrieved_context
        )

## 43. Initialize the Assistant

In [43]:
assistant = AIRetailAnalyticsAssistant(
    planner=planner,
    sql_generator=sql_generator,
    sql_executor=sql_executor,
    knowledge_retriever=knowledge_retriever,
    response_generator=response_generator
)

## 44. Test SQL Query

In [51]:
response = assistant.ask(
    "Which region generated the highest revenue?"
)

print(response)

{
    "needs_sql": true,
    "needs_rag": false,
    "sql_task": "Determine the region with the highest total revenue.",
    "rag_task": "",
    "answer_goal": "The region that generated the highest revenue."
}
Based on the provided information, the South region generated the highest revenue with a total of $10,724,693.11.


## 45. Test RAG Query

In [54]:
response = assistant.ask(
    "What is Promotion Lift?"
)

print(response)

{
    "needs_sql": false,
    "needs_rag": true,
    "sql_task": "",
    "rag_task": "Retrieve the definition and explanation of Promotion Lift from the Knowledge Base.",
    "answer_goal": "A definition and explanation of what Promotion Lift is in the context of retail analytics."
}
Promotion Lift is a metric used in retail analytics to measure the effectiveness of a promotional activity. It is calculated as the difference between the revenue generated during the promotion period and the baseline revenue, divided by the baseline revenue.

Here’s a breakdown of the key components:

- **Current Promotion Revenue**: This is the total revenue generated during the week(s) the promotion was active. In SQL terms, it would be the sum of `revenue` for the product or region where the promotion was active (`promotion_active = True`).

- **Baseline Revenue**: This is the average weekly revenue for the same product or region over the 4 weeks immediately preceding the promotion's start date. It ser

## 46. Test Hybrid Query

In [ ]:
response = assistant.ask(
    "Explain whether promotions improved sales in the South region."
)

print(response)

{
    "needs_sql": true,
    "needs_rag": true,
    "sql_task": "Calculate the total sales and number of promotions in the South region before and after a specific period to compare sales trends.",
    "rag_task": "Define the criteria for what constitutes a promotion and explain the retail KPIs relevant to measuring the impact of promotions on sales.",
    "answer_goal": "An analysis indicating whether promotions had a positive impact on sales in the South region, including sales figures and promotion details."
}
Based on the provided data, we can analyze whether promotions improved sales in the South region. Here's a summary of the available information:

- **Total Sales in South Region**: $369,103.01
- **Number of Promotions in South Region**: 3

To determine if promotions had a positive impact on sales, we would typically need more detailed information such as:

1. **Sales figures before and during the promotions** to calculate the promotion lift.
2. **Baseline revenue** (sales in t

# Custom Queries

In [53]:
print("How can I help you?")

users_question = input("Type your query...")

response = assistant.ask(users_question)

print(response)

How can I help you?
{
    "needs_sql": false,
    "needs_rag": true,
    "sql_task": "",
    "rag_task": "Retrieve the definition and formula for baseline revenue from the Knowledge Base.",
    "answer_goal": "The formula for baseline revenue should be described, including any necessary components or context."
}
The formula for baseline revenue is derived as the average weekly revenue for the same product/region over the 4 weeks immediately preceding the promotion's start date. Here's how you can calculate it:

1. **Identify the Promotion Start Date**: Determine the start date of the promotion.
2. **Select the Relevant Period**: Look at the 4 weeks immediately before the promotion's start date.
3. **Calculate Weekly Revenue**: For each of these 4 weeks, sum up the `revenue` for the product/region.
4. **Compute the Average**: Divide the total revenue from these 4 weeks by 4 to get the baseline revenue.

Mathematically, if \( R_i \) represents the revenue for the \( i \)-th week in the 4